In [ ]:
# Philipp Bartz 2025

#import serial

import matplotlib.pyplot as plt
import os
import velox
import time
import custom_pathlibrary as cpath
import custom_filehandler as cfile
import custom_measurement as cmeasure
import custom_waferprober as cwafer
import custom_logging as clog
import custom_analyzer as canal
import logging
import matplotlib.pyplot as plt
import serial 

# EXAMPLE
# print(f"{bcolors.QUIET}{bcolors.BOLD}Warning: No active frommets remain. Continue?{bcolors.ENDC}")

settings = cfile.save_to_json(folder="")

In [ ]:
def is_running(*, logger):
    try:
        with open("activate.txt", 'r') as f:
            contents = f.read()
        if "True" in contents:
            return True
        else:
            logger.critical("activate.txt does not contain 'True'. Stopping the program.")
            return False
    except FileNotFoundError:
        logger.critical("File not found: activate.txt")
        return False
    except Exception as e:
        logger.critical(f"Error reading activate.txt: {e}")
        return False

# Main control loop
def main(*, settings, 
         only_init=False, 
         only_single_subdie=False, 
         chuck_target_temperature=25, # target temperature for the chuck in degrees Celsius, set to 25 for room temperature
         force_temperature=True, 
         extra="",  # additional information for the pathlibrary, used for alternative path configurations For more information, see the custom_pathlibrary.py file and README.md
         tag_1="", 
         tag_2="", 
         tag_3="", 
         measurements_per_diode=1   
         ):


    ###     test parameters     ###
    waferprober_ip = "192.168.255.1"     # IP address of the wafer prober
    device_port = "/dev/ttyUSB0"
    plotting = False        # if you want to plot the collected IV Data directly after the measurement, turn this to True
    movement_time = 0   # time in seconds we grant the prober to move to the next position
    
    
    ###     Initialization     ###
    wafer_folder = cfile.create_folder()
    timestr = time.strftime("%Y_%m_%d-%H_%M_%S")
    tmptime = time.time()

    try:    # this should kill loggers from previous runs and avoid logging in the wrong folder
        logging.shutdown()
    except:
        pass
    logger = clog.CustomLogger.setup_logger(name="custom_logger", log_file=f"{wafer_folder}/event.log")
    logger.critical(f"Start time: {timestr}")

    settings.add(name="time_at_start", value=timestr)
    settings.add(name="device_port", value=device_port)
    settings.add(name="waferprober_ip", value=waferprober_ip)
    settings.add(name="plotting", value=plotting)
    settings.add(name="movement_time", value=movement_time)
    settings.add(name="chuck_target_temperature", value=chuck_target_temperature)
    settings.add(name="extra", value=extra)
    settings.add(name="wafer_folder", value=wafer_folder)
    settings.add(name="tag_1", value=tag_1)
    settings.add(name="tag_2", value=tag_2)
    settings.add(name="tag_3", value=tag_3)
    settings.add(name="measurements_per_diode", value=measurements_per_diode)
    
    if not is_running(logger=logger):
        return

    # initialize Keithley
    cmeasure.init_IV(dev=device_port, logger=logger)

    # Connect to Velox Message Server
    msgServer = cwafer.connect_to_message_server(ip=waferprober_ip, logger=logger)
    if not msgServer:
        logger.error("Could not connect to Velox Message Server.")
        return
    if only_init:
        return msgServer, logger
    
    cwafer.register_applications(logger=logger)

    
    cwafer.set_heater_temp(msgServer=msgServer, temperature=chuck_target_temperature, logger=logger)
    if force_temperature:
        current_temperature = cwafer.get_chuck_temperature(msgServer=msgServer, logger=logger)
        while current_temperature != chuck_target_temperature:
            time.sleep(30) # wait for 30 seconds before checking again if the chuck has reached the target temperature
            current_temperature = cwafer.get_chuck_temperature(msgServer=msgServer, logger=logger)

    if wafer_folder is None:
        logger.error("Wafer folder is not created. Exiting..")
        return
    
    ###     End of initialization     ###


    
    while cwafer.step_to_next_subdie(msgServer=msgServer, logger=logger) and is_running(logger=logger):   
        logger.info(f"Starting new subdie movement and measurement")
        path = cpath.PLib()
        pathsteps = path.find_path(msgServer=msgServer,extra=extra, logger=logger)
        for i in pathsteps:
            probe = msgServer.sendSciCommand("MoveProbeSeparation", 1)
            logger.path(f"{probe} on seperation height") # type: ignore
            time.sleep(movement_time)
            logger.path(f"Moving Probe to {i[0]}, {i[1]}") # type: ignore
            msgServer.sendSciCommand("MoveProbe", 1, i[0], i[1])
            time.sleep(movement_time)
            if not is_running(logger=logger):
                logger.critical("Stopping Path due to is_running check.")
                break
            
            if i[2] == "PAD":   # if the pathpoint is a tagged PAD, it is a measurement point
                diode_Nr = i[3]
                logger.lowering(f"Lowering Probe to contact Hight") # type: ignore
                msgServer.sendSciCommand("MoveProbeContact", 1)

                cwafer.set_quiet_mode(msgServer=msgServer, quiet_mode=True, logger=logger)
                logger.measurement(f"STARTING MEASUREMENT of diode {diode_Nr}") # type: ignore

            ###          MEASUREMENT        ###
                for i in range(measurements_per_diode):
                    measurement_no = i + 1
                    logger.measurement(f"Measurement {measurement_no} of {measurements_per_diode}") # type: ignore
                    cmeasure.measure(msgServer=msgServer, device_port=device_port, folder=wafer_folder, diode_Nr=diode_Nr, settings=settings, logger=logger, plotting=plotting, extra=extra, tag_1=tag_1, tag_2=tag_2, tag_3=tag_3, measurements_per_diode=measurements_per_diode, measurement_no=measurement_no)
                    logger.measurement(f"MEASUREMENT DONE") # type: ignore
                    if (tmptime + 3600) < time.time():
                        tmptime = time.time()
                        print("still running")
                        logger.critical(f"Waferprober still running. Measurement no {i} of {measurements_per_diode} at diode_Nr {diode_Nr}")
                    if not is_running(logger=logger):
                        logger.critical("Stopping Measurement due to is_running check.")
                        break
            ###          END OF MEASUREMENT        ###

                cwafer.set_quiet_mode(msgServer=msgServer, quiet_mode=False, logger=logger)
        logger.path(f"ALL MOVEMENTS DONE for this Subdie")
        if only_single_subdie:
            logger.info("Only single subdie measurement requested, stopping after this subdie.")
            break
        if not is_running(logger=logger):
            logger.critical("Stopping Measurement Loop due to is_running check.")
            break


###     END OF PROBE AND MEASURE     ###)
    logger.info(f"Probe and measure completed successfully.")
    timestr = time.strftime("%Y_%m_%d-%H_%M_%S")
    settings.add(name="time_at_end", value=timestr)
    settings.save(folder=wafer_folder, logger=logger)
    



print("init finished")

In [ ]:
## test single subdie at static temperature of 25C 
main(settings=settings, only_single_subdie = True)


In [ ]:
#### Script for Testing waferprober at different Temperatures ####
# this tests the wafer from 30 to 90 degrees Celsius in steps of 5 degrees
for i in range(30, 95, 5):
    main(settings=settings, only_init=False, only_single_subdie=False, chuck_target_temperature=i, force_temperature=True, tag_1="temperature_test")

In [ ]:
msgServer, logger = main(settings=settings, only_init=True)


In [ ]:
device_port = "/dev/ttyUSB0"
iv_data = []
#cmeasure.init_IV(dev=device_port, logger=logger)
try:
        with serial.Serial(device_port, baudrate=57600)  as ser:
            ser.write(b"*RST\n")
            ser.write(b"CURR:RANG 2E-9\n")
            ser.write(b"SYST:ZCH ON\n")
            ser.write(b"ARM:COUN 1\n")
            ser.write(b"INIT\n")
            ser.write(b"CURR:RANG:AUTO ON\n")
            ser.write(b"SOUR:VOLT:RANG 500\n")
            ser.write(b"SOUR:VOLT:ILIM 2.50e-3\n")
            ser.write(b"FORM:ELEM READ,VSO\n")
            ser.write(b"SYST:ZCOR:STAT OFF\n")
            ser.write(b"SYST:ZCOR:ACQ\n")
            ser.write(b"SYST:ZCOR:STAT ON\n")
            logger.info("Keithley initialized successfully.")

except Exception as e:
        logger.critical(f"Error initializing Keithley: {e}")

In [ ]:
msgServer.sendSciCommand("StepToDie", 177)

In [ ]:
import time
tmptime = time.time()
while is_running(logger="logger"):
    print("^")
    time.sleep(2)
    if tmptime + 60  < time.time():
        tmptime = time.time()
        print("still running")

print("stopped")